In [1]:
# Minimal "plot.py-like" visualization for ONE Property Prediction task (BACE/BBBP/ClinTox/HIV/Tox21)
# Uses Llama-3.1-8B-Instruct and produces the same 2x2 figures (full attn + ROI + bin + denoised).

import os, re, json
os.environ["CUDA_VISIBLE_DEVICES"] = "1,2,3"
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from transformers import AutoTokenizer, AutoModelForCausalLM

import sys
from pathlib import Path

GRAPHLENS_SRC = Path("/home/lym/LLM-Research/Attention/Graph_Attention/src/GraphLens/src")
if str(GRAPHLENS_SRC) not in sys.path:
    sys.path.insert(0, str(GRAPHLENS_SRC))

# quick sanity check
import graphlens
print("graphlens imported from:", graphlens.__file__)

# GraphLens plotting/helpers (same ones used by plot.py)
from graphlens.utils import (
    build_sawtooth_mask,
    extract_roi_from_attn,
    preprocess_for_scoring,
    normalize_minmax,
)
from graphlens.viz_utils import (
    create_layer_figure,
    create_head_figure,
)

# -----------------------
# Config (edit these)
# -----------------------
TASK = "BBBP"  # one of: BACE BBBP ClinTox HIV Tox21
DATA_DIR = "/home/lym/LLM-Research/Attention/Graph_Attention/src/GraphLens/baselines/repos/ChemLLMBench/data/property_prediction"
PROMPT_PATH = os.path.join(DATA_DIR, "property_prediction_prompt.txt")

MODEL_PATH = "/home/lym/data1/LLM-model/meta-llama/Meta-Llama-3.1-8B-Instruct"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Pick which visualization
PLOT_MODE = "layer"  # "layer" or "head"
LAYER_ID = 10
HEAD_ID = 5

# scoring/denoise params (same defaults as GraphLens)
BINARIZE_METHOD = "topk"
PRE_THRESHOLD_FRAC = 0.1
DPI = 220
MAX_SEQ_LEN = 1024

# -----------------------
# Load base prompt from txt
# -----------------------
def load_prompt_map(prompt_txt_path: str):
    txt = open(prompt_txt_path, "r", encoding="utf-8").read()
    pat = re.compile(r"^\s*([A-Za-z0-9_-]+)\s*:\s*prompt\s*=\s*\"([\s\S]*?)\"\s*$", re.MULTILINE)
    out = {}
    for m in pat.finditer(txt):
        k = m.group(1).strip()
        p = m.group(2).replace("\\n", "\n").replace("\\\"", "\"").strip()
        out[k] = p
    return out

PROMPTS = load_prompt_map(PROMPT_PATH)
base_prompt = PROMPTS[TASK]

# -----------------------
# Load one sample (test preferred; otherwise any csv)
# -----------------------
def resolve_csv(task: str):
    for a, b in [(f"{task}_test.csv", f"{task}_train.csv"),
                 (f"{task.lower()}_test.csv", f"{task.lower()}_train.csv")]:
        tp = os.path.join(DATA_DIR, a)
        tr = os.path.join(DATA_DIR, b)
        if os.path.exists(tp):
            return tp
        if os.path.exists(tr):
            return tr
    raise FileNotFoundError(f"Cannot find {task}_test.csv or {task}_train.csv under {DATA_DIR}")

csv_path = resolve_csv(TASK)
df = pd.read_csv(csv_path)

# figure out smiles column for each task (per your headers)
SMILES_COL = {
    "BACE": "mol",
    "BBBP": "smiles",
    "ClinTox": "smiles",
    "HIV": "smiles",
    "Tox21": "smiles",
}[TASK]

# pick first non-empty smiles
row = None
for i in range(len(df)):
    s = str(df.loc[i, SMILES_COL]) if SMILES_COL in df.columns else ""
    if isinstance(s, str) and s.strip():
        row = df.loc[i]
        break
assert row is not None, "No valid SMILES found"

smiles = str(row[SMILES_COL]).strip()

# -----------------------
# Build a simple prompt (zero-shot)
# (We only need a stable "SMILES: ...\n<field>:" line for span detection)
# -----------------------
def build_query(task: str, smiles: str):
    if task == "BACE":
        return f"SMILES: {smiles}\nBACE-1 Inhibit:"
    if task == "BBBP":
        return f"SMILES: {smiles}\nBBBP Penetration:"
    if task == "ClinTox":
        return f"SMILES: {smiles}\nClinically-trial-toxic:"
    if task == "HIV":
        return f"SMILES: {smiles}\nHIV Inhibit:"
    if task == "Tox21":
        # simplest: choose one assay name to visualize prompt structure
        return f"SMILES: {smiles}\nAssay: NR-AR\nToxic:"
    return f"SMILES: {smiles}\nLabel:"

prompt = base_prompt.strip() + "\n\n" + build_query(TASK, smiles) + "\n"
print(prompt)

# -----------------------
# Find token span that corresponds to the SMILES substring in the prompt
# (This replaces GraphWiz edge-span logic.)
# -----------------------
def get_smiles_token_span(prompt: str, smiles: str, tokenizer):
    # find char span of the first occurrence of smiles
    start = prompt.find(smiles)
    if start < 0:
        raise ValueError("SMILES not found in prompt (unexpected).")
    end = start + len(smiles)

    enc = tokenizer(prompt, return_tensors="pt", return_offsets_mapping=True, add_special_tokens=True)
    offsets = enc["offset_mapping"][0].tolist()  # list[(s,e)]

    t_start, t_end = None, None
    for ti, (s, e) in enumerate(offsets):
        if s == 0 and e == 0:
            continue
        if e > start:
            t_start = ti
            break
    for ti in range(len(offsets) - 1, -1, -1):
        s, e = offsets[ti]
        if s == 0 and e == 0:
            continue
        if s < end:
            t_end = ti
            break

    if t_start is None or t_end is None or t_end < t_start:
        raise ValueError("Failed to map SMILES char-span to token-span.")
    return t_start, t_end

# -----------------------
# Load model and run attentions
# -----------------------
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, padding_side="left")
if tokenizer.pad_token_id is None and tokenizer.eos_token_id is not None:
    tokenizer.pad_token_id = tokenizer.eos_token_id

model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    device_map="auto",
    torch_dtype="auto",
    attn_implementation="eager",
)
model.eval()

enc = tokenizer(prompt, return_tensors="pt").to(model.device)
if enc.input_ids.shape[1] > MAX_SEQ_LEN:
    enc.input_ids = enc.input_ids[:, :MAX_SEQ_LEN]
    if "attention_mask" in enc:
        enc.attention_mask = enc.attention_mask[:, :MAX_SEQ_LEN]

with torch.inference_mode():
    out = model(enc.input_ids, output_attentions=True, use_cache=False)

# stack attentions: [L, H, S, S]
attn_np = np.stack([a[0].to(torch.float32).detach().cpu().numpy() for a in out.attentions], axis=0)

num_layers, num_heads, S, _ = attn_np.shape
print("attn shape:", attn_np.shape)


# -----------------------
# Build ROI around the SMILES token span
# -----------------------
t_start, t_end = get_smiles_token_span(prompt, smiles, tokenizer)
g_start, g_end = t_start, t_end
span_len = g_end - g_start + 1

# one local span covering the full SMILES area -> sawtooth mask becomes a lower-triangle
local_spans = [(0, span_len - 1)]
ideal_mask = build_sawtooth_mask(span_len, local_spans)


# ---- Save per-layer figures to local ----
import os
import matplotlib.pyplot as plt
import numpy as np

SAVE_DIR = f"./pp_plots/{TASK}_llama3.1-8b/ROI_smiles"
os.makedirs(SAVE_DIR, exist_ok=True)

print("saving to:", os.path.abspath(SAVE_DIR))

for l in range(num_layers):
    # avg ROI over heads
    rois = []
    full_heads = []
    for h in range(num_heads):
        rois.append(extract_roi_from_attn(attn_np, l, h, g_start, g_end))
        full_heads.append(attn_np[l, h])

    avg_roi = np.mean(np.stack(rois, axis=0), axis=0)

    bin_mask, denoised = preprocess_for_scoring(
        avg_roi,
        binarize_method=BINARIZE_METHOD,
        ideal_mask=ideal_mask,
        pre_threshold_frac=PRE_THRESHOLD_FRAC,
    )

    full_attn_layer = np.mean(np.stack(full_heads, axis=0), axis=0)

    title = f"{TASK} | L{l} AvgHeads | ROI=SMILES tokens [{g_start},{g_end}]"
    fig = create_layer_figure(
        full_attn_layer=full_attn_layer,
        avg_roi=avg_roi,
        bin_mask=bin_mask,
        blurred_norm=denoised,
        title=title,
        local_spans=local_spans,
        g_start=g_start,
        g_end=g_end,
    )

    out_png = os.path.join(SAVE_DIR, f"{TASK}_L{l:02d}_avgheads.png")
    fig.savefig(out_png, dpi=DPI, bbox_inches="tight")
    plt.close(fig)

print("done.")

# # -----------------------
# # Plot: "layer" mode (average heads) or "head" mode (single head)
# # -----------------------
# if PLOT_MODE == "layer":
#     l = int(min(max(LAYER_ID, 0), num_layers - 1))

#     # avg ROI over heads
#     rois = []
#     full_heads = []
#     for h in range(num_heads):
#         roi = extract_roi_from_attn(attn_np, l, h, g_start, g_end)
#         rois.append(roi)
#         full_heads.append(attn_np[l, h])

#     avg_roi = np.mean(np.stack(rois, axis=0), axis=0)
#     bin_mask, denoised = preprocess_for_scoring(
#         avg_roi,
#         binarize_method=BINARIZE_METHOD,
#         ideal_mask=ideal_mask,
#         pre_threshold_frac=PRE_THRESHOLD_FRAC,
#     )

#     # full attention layer-avg
#     full_attn_layer = np.mean(np.stack(full_heads, axis=0), axis=0)

#     title = f"{TASK} | L{l} AvgHeads | ROI=SMILES tokens [{g_start},{g_end}]"
#     fig = create_layer_figure(
#         full_attn_layer=full_attn_layer,
#         avg_roi=avg_roi,
#         bin_mask=bin_mask,
#         blurred_norm=denoised,
#         title=title,
#         local_spans=local_spans,
#         g_start=g_start,
#         g_end=g_end,
#     )
#     fig.set_dpi(DPI)
#     plt.show()

# elif PLOT_MODE == "head":
#     l = int(min(max(LAYER_ID, 0), num_layers - 1))
#     h = int(min(max(HEAD_ID, 0), num_heads - 1))

#     roi = extract_roi_from_attn(attn_np, l, h, g_start, g_end)
#     bin_mask, denoised = preprocess_for_scoring(
#         roi,
#         binarize_method=BINARIZE_METHOD,
#         ideal_mask=ideal_mask,
#         pre_threshold_frac=PRE_THRESHOLD_FRAC,
#     )

#     title = f"{TASK} | L{l} H{h} | ROI=SMILES tokens [{g_start},{g_end}]"
#     fig = create_head_figure(
#         full_attn_head=attn_np[l, h],
#         roi=roi,
#         bin_mask=bin_mask,
#         blurred_norm=denoised,
#         title=title,
#         local_spans=local_spans,
#         g_start=g_start,
#         g_end=g_end,
#     )
#     fig.set_dpi(DPI)
#     plt.show()

# else:
#     raise ValueError("PLOT_MODE must be 'layer' or 'head'")

graphlens imported from: /home/lym/LLM-Research/Attention/Graph_Attention/src/GraphLens/src/graphlens/__init__.py
You are an expert chemist, your task is to predict the property of molecule using your experienced chemical property prediction knowledge. 
Please strictly follow the format, no other information can be provided. Given the SMILES string of a molecule, the task focuses on predicting molecular properties, specifically penetration/non-penetration to the brain-blood barrier, based on the SMILES string representation of each molecule. You will be provided with several examples molecules, each accompanied by a binary label indicating whether it has penetrative property (Yes) or not (No). The task is to predict the binary label for a given molecule, please answer with only Yes or No.

SMILES: C1=C(Cl)C=CC2=C1C(=NCC(=O)N2CC(F)(F)F)C3=CC=CC=C3
BBBP Penetration:



Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

attn shape: (32, 32, 174, 174)
saving to: /home/lym/LLM-Research/Attention/Graph_Attention/src/GraphLens/baselines/molecularNet/pp_plots/BBBP_llama3.1-8b/ROI_smiles
done.
